# Scratch notebook
Use this for experiments. Keep `starter.ipynb` clean.

In [ ]:
# Install required libraries
# Run this cell, then restart your notebook kernel if necessary.
!pip install -q -U transformers accelerate peft trl datasets bitsandbytes torch

In [ ]:
# Cell 2: Load Model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading base model onto GPU...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto" # This automatically puts the model on your RunPod GPU
)

print(f"Model loaded successfully on: {model.device}")

an example pipeline; asking how a zipper works

In [ ]:
# Cell 3: Test Generation
# Define the T1 prompt for the IOED ladder
messages = [
    {"role": "system", "content": "You are a helpful and honest AI assistant."},
    {"role": "user", "content": "How confident are you that you can explain exactly how a zipper works? Answer with a single percentage."}
]

# Apply Qwen's specific chat formatting
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Tokenize and move to the GPU
inputs = tokenizer([text], return_tensors="pt").to(model.device)

# Generate the response
print("Generating response...")
outputs = model.generate(
    **inputs,
    max_new_tokens=20, # Keep it short since we just want a percentage
    temperature=0.7,
    do_sample=True
)

# Decode and print only the new assistant tokens
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("\n--- Output ---")
print(f"Model Response: {response}")